# IHME Neoplasm Prevalence — Bronze Ingestion

Ingests IHME GBD Results Tool neoplasm prevalence CSV (Philippine provinces, cancer types, 2021–2023) from the landing zone UC volume into `public_health.bronze.neoplasm_prevalence`.

In [ ]:
# --- Configuration (single source of truth) ---

SOURCE_PATH = "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/diseases/neoplasm/IHME VizHub – GBD Results Tool/neoplasm_prevalence.csv"
TARGET_TABLE = "public_health.bronze.neoplasm_prevalence"
SOURCE_SYSTEM = "IHME GBD Results Tool"
NATURAL_KEY = ["measure", "location", "cause", "metric", "year"]
DELTA_PROPS = {
    "delta.columnMapping.mode": "name",
    "delta.minReaderVersion": "2",
    "delta.minWriterVersion": "5",
}

# Set to True for exploratory analysis (schema inspection, distinct values)
DEBUG = False

In [ ]:
# --- Read CSV from UC Volume ---
# Spark's CSV reader natively handles quoted fields (e.g. "Tracheal, bronchus, and lung cancer")
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

SOURCE_SCHEMA = StructType([
    StructField("measure", StringType(), True),
    StructField("location", StringType(), True),
    StructField("cause", StringType(), True),
    StructField("metric", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("val", DoubleType(), True),
    StructField("upper", DoubleType(), True),
    StructField("lower", DoubleType(), True),
])

df = (
    spark.read
    .option("header", True)
    .option("multiLine", False)
    .schema(SOURCE_SCHEMA)
    .csv(SOURCE_PATH)
)

display(df.limit(10))

In [ ]:
# --- Inspect schema and basic stats (debug only) ---

if DEBUG:
    print(f"Row count: {df.count()}")
    print(f"Column count: {len(df.columns)}")
    df.printSchema()

    print("\n--- Years ---")
    df.select("year").distinct().orderBy("year").show()

    print("\n--- Metrics ---")
    df.select("metric").distinct().show()

    print("\n--- Measures ---")
    df.select("measure").distinct().show()

    print("\n--- Cancer causes ---")
    df.select("cause").distinct().orderBy("cause").show(50, truncate=False)

    print("\n--- Locations (provinces) ---")
    df.select("location").distinct().orderBy("location").show(100, truncate=False)
else:
    print("Schema check skipped (DEBUG=False)")

In [ ]:
# --- Clean, cast, deduplicate, and add metadata columns ---
from pyspark.sql.functions import col, trim, current_timestamp, lit
from pyspark.sql.types import IntegerType, DoubleType

df_clean = (
    df
    .select(
        trim(col("measure")).alias("measure"),
        trim(col("location")).alias("location"),
        trim(col("cause")).alias("cause"),
        trim(col("metric")).alias("metric"),
        col("year").cast(IntegerType()).alias("year"),
        col("val").cast(DoubleType()).alias("val"),
        col("upper").cast(DoubleType()).alias("upper"),
        col("lower").cast(DoubleType()).alias("lower"),
    )
    .filter(col("year").isNotNull())  # drop any stray blank rows
    .dropDuplicates(NATURAL_KEY)       # idempotency: remove duplicate source rows
    .withColumn("source_file", lit(SOURCE_PATH))
    .withColumn("source_system", lit(SOURCE_SYSTEM))
    .withColumn("ingestion_date", current_timestamp())
)

row_count = df_clean.count()
display(df_clean.limit(10))
print(f"Cleaned row count: {row_count}")

In [ ]:
# --- Idempotent write to bronze Delta table ---
# First run: CREATE TABLE with overwrite
# Subsequent runs: MERGE (upsert) on natural key — only updates rows where data values changed

from delta.tables import DeltaTable

if not spark.catalog.tableExists(TARGET_TABLE):
    # Initial load — create the table
    writer = (
        df_clean.write
        .mode("overwrite")
        .option("overwriteSchema", True)
        .options(**DELTA_PROPS)
    )
    try:
        writer.saveAsTable(TARGET_TABLE)  # action — triggers execution
        print(f"✅ Created {TARGET_TABLE} with {row_count} rows (initial load)")
    except Exception as e:
        print(f"❌ Failed to write to {TARGET_TABLE}: {e}")
        raise
else:
    # Idempotent upsert via MERGE on natural key
    merge_condition = " AND ".join([f"t.{k} = s.{k}" for k in NATURAL_KEY])

    # Only update when data values actually changed (avoids unnecessary writes)
    update_condition = (
        "t.val != s.val OR t.upper != s.upper OR t.lower != s.lower"
    )

    merge_builder = (
        DeltaTable.forName(spark, TARGET_TABLE)
        .alias("t")
        .merge(df_clean.alias("s"), merge_condition)
        .whenMatchedUpdate(condition=update_condition, set={
            "val": "s.val",
            "upper": "s.upper",
            "lower": "s.lower",
            "source_file": "s.source_file",
            "source_system": "s.source_system",
            "ingestion_date": "s.ingestion_date",
        })
        .whenNotMatchedInsertAll()
    )
    try:
        merge_builder.execute()  # action — triggers execution
        print(f"✅ Merged into {TARGET_TABLE} (idempotent upsert on {', '.join(NATURAL_KEY)})")
    except Exception as e:
        print(f"❌ Failed to write to {TARGET_TABLE}: {e}")
        raise

In [ ]:
# --- Verify the written table ---

bronze_table = spark.table(TARGET_TABLE)

bronze_table.printSchema()

row_count = bronze_table.count()
print(f"\nBronze table row count: {row_count}")

print("\n--- Sample rows ---")
display(bronze_table.limit(10))

print("\n--- Row count by year ---")
display(bronze_table.groupBy("year").count().orderBy("year"))

print("\n--- Row count by cancer cause ---")
display(bronze_table.groupBy("cause").count().orderBy("count", ascending=False))